# Training phase

Trains each of the 6 policies (3 discrete tabular methods, 3 continuous methods) with fixed hyperparameters and saves each one to `saved_policies/<name>/`. Run cells individually per algorithm, or run the whole notebook top to bottom.

Model testing / success-criterion evaluation lives in `test_models.ipynb`, not here.

In [ ]:
import sys
!git clone https://github.com/NomeMio/rl_inverse_pendulum.git

sys.path.insert(0, "/content/rl_inverse_pendulum")
    
!pip install -r /content/rl_inverse_pendulum/requirements.txt
!pip install gymnasium[mujoco]

In [ ]:
from cart_model import Cart_model, Episode
from discrete_policies import SarsaAgent, QLearningAgent, ExpectedSarsaAgent
from continuous_policies import ReinforcePolicy
from train import train_step_policy, train_episodic_policy

context = Cart_model(human=False)


## SARSA (discrete)

In [ ]:
sarsa_policy = SarsaAgent(gamma=0.98, alpha=0.1, epsilon_start=0.5, epsilon_min=0.05,
                           n_state_buckets=7, n_action_buckets=9)
train_step_policy(sarsa_policy, context, episodes=8000, max_steps=1000, desc="sarsa")
sarsa_policy.save("saved_policies/sarsa")

## Q-learning (discrete)

In [ ]:
q_learning_policy = QLearningAgent(gamma=0.98, alpha=0.1, epsilon_start=0.5, epsilon_min=0.05,
                                    n_state_buckets=7, n_action_buckets=9)
train_step_policy(q_learning_policy, context, episodes=8000, max_steps=1000, desc="q_learning")
q_learning_policy.save("saved_policies/q_learning")

## Expected SARSA (discrete)

In [ ]:
expected_sarsa_policy = ExpectedSarsaAgent(gamma=0.98, alpha=0.1, epsilon_start=0.5, epsilon_min=0.05,
                                            n_state_buckets=7, n_action_buckets=9)
train_step_policy(expected_sarsa_policy, context, episodes=8000, max_steps=1000, desc="expected_sarsa")
expected_sarsa_policy.save("saved_policies/expected_sarsa")

## REINFORCE (discrete actions, softmax-in-action-preferences)

Two separate grid searches, one per `weight_mode` (pinned via `fixed_kwargs` so
it isn't swept -- the two modes have different weight shapes/semantics, mixing
them in one Cartesian product wouldn't make sense): `"per_action"` (a weight
row per action over state features) and `"shared"` (one weight vector over
state*action interaction features). `fixed_kwargs` always instantiates the
policy with those exact kwargs alongside whatever the grid is searching over.

In [ ]:
reinforce_per_action_best_params, reinforce_per_action_policy, reinforce_per_action_best_metrics, reinforce_per_action_search_results = (
    ReinforcePolicy.grid_search(
        context,
        fixed_kwargs={"weight_mode": "per_action"},
        train_episodes=3000,       # per-combo search budget (smaller than a full training run)
        max_steps=1000,
        eval_episodes=50,
        final_train_episodes=8000,  # retrain the winning combo from scratch at this larger budget
        desc="reinforce_per_action",
    )
)
print("reinforce (per_action) best params:", reinforce_per_action_best_params)
print("reinforce (per_action) best metrics:", reinforce_per_action_best_metrics)
reinforce_per_action_policy.save("saved_policies/reinforce_per_action")

In [ ]:
reinforce_shared_best_params, reinforce_shared_policy, reinforce_shared_best_metrics, reinforce_shared_search_results = (
    ReinforcePolicy.grid_search(
        context,
        fixed_kwargs={"weight_mode": "shared"},
        train_episodes=1500,       # per-combo search budget (smaller than a full training run)
        max_steps=1000,
        eval_episodes=50,
        final_train_episodes=8000,  # retrain the winning combo from scratch at this larger budget
        desc="reinforce_shared",
    )
)
print("reinforce (shared) best params:", reinforce_shared_best_params)
print("reinforce (shared) best metrics:", reinforce_shared_best_metrics)
reinforce_shared_policy.save("saved_policies/reinforce_shared")

### Show the calculated policy

Heatmap of the chosen action across `pole_angle` x `pole_angle_velocity` (cart
position/velocity held at 0) for each of the two trained REINFORCE policies --
a sane balancing policy should show a clear push-left/push-right split roughly
antisymmetric through the origin.

In [ ]:
import matplotlib.pyplot as plt
from visualize_policy import policy_action_heatmap

policy_action_heatmap(reinforce_per_action_policy)
plt.show()

policy_action_heatmap(reinforce_shared_policy)
plt.show()

In [ ]:
context.close()